# SHMQ-Ultimate: 3-Level Quantization Framework for Qwen2.5-7B-Instruct on T4

This notebook implements the full SHMQ-Ultimate quantization pipeline (7 sources integrated)
and benchmarks three models on a single T4 GPU (16GB, sm_75):

1. **Baseline (FP16)** — Qwen2.5-7B-Instruct unmodified
2. **MixLLM original** — Microsoft's W4.4A8 (2-level, 90% INT4 + 10% INT8)
3. **SHMQ-Ultimate (ours)** — 3-level {5% FP16 + 20% INT8 + 75% INT4} with full pipeline

## Architecture (7 sources integrated)

| Component | Source | Role |
|-----------|--------|------|
| PyHessian + Fisher | HAWQ-V3 | Inter-layer sensitivity (SHMQ Eq.6) |
| GPTQ OBS | SliM-LLM | Per-element Hessian (SHMQ Eq.5) |
| ILP solver {4,8,16} | HAWQ-V3 + custom | Bit allocation (3-level) |
| ISA matching | PolyQ | Tensor-core tile rounding (128/64) |
| Decoupled permutation | SHMQ paper Eq.12 | Cluster C16/C8/C4 sort |
| RMSNorm fusion | SHMQ paper §3.2 | Permutation absorbed by norm |
| AutoRound | Intel | Learnable rounding (SignSGD, 200 steps) |
| SmoothQuant | MIT-Han-Lab | Activation outlier migration |
| SQC | SliM-LLM | Salience-weighted quantizer calibration |
| 3-level CUDA kernel | **Custom (cupy.RawKernel)** | Fused FP16+INT8+INT4 GEMM, T4 sm_75 |
| vLLM inference | MixLLM + custom patch | T4 support, 3-level method |

## Pipeline (11 steps)

1. SmoothQuant pre-processing
2. PyHessian inter-layer sensitivity (HAWQ-V3)
3. OBS per-element sensitivity (SliM-LLM)
4. ILP bit allocation {4,8,16} (HAWQ-V3, PULP)
5. ISA-aware quanta matching (PolyQ)
6. Decoupled permutation on 3 clusters (SHMQ Eq.12)
7. RMSNorm fusion + layout propagation (PolyQ §4)
8. AutoRound 200-step SignSGD
9. SQC calibration
10. GPTQ + mixed INT4/INT8/FP16 quantization
11. SHMQ 3-level CUDA kernel inference

## T4 (sm_75) compatibility

The custom 3-level CUDA kernel is compiled at runtime via `cupy.RawKernel` + NVRTC,
using Turing-specific PTX:
  - FP16: `mma.sync.aligned.m16n8k16.row.col.f32.f16.f16.f32`
  - INT8: `mma.sync.aligned.m8n8k16.row.col.s32.s8.s8.s32`
  - INT4: `mma.sync.aligned.m8n8k4.row.col.s32.s4.s4.s32`

**Estimated runtime on T4**: ~3.5 hours per model (quantization + eval), 7-10h for all 3.

**Expected results**:
  - WikiText-2 PPL: ~7.55 (FP16) → ~7.58 (SHMQ-Ultimate), gap < 0.5%
  - Inference speedup: 2.0-2.5x vs FP16 on T4
  - Memory footprint: 14.3 GB → 5.4 GB (3.21x compression)

## Cell 1: Environment setup

Installs all required packages. Run this once.

In [ ]:
# Cell 1: Install dependencies (run once, then restart kernel)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate
!pip install -q pulp            # ILP solver (HAWQ-V3)
!pip install -q cupy-cuda12x    # NVRTC runtime kernel compilation
!pip install -q datasets evaluate
!pip install -q lm-eval         # zero-shot benchmarks
!pip install -q matplotlib pandas seaborn
!pip install -q vllm==0.9.0     # production inference (only if doing vLLM benchmark)

# Verify GPU
import torch
assert torch.cuda.is_available(), 'CUDA not available — T4 required'
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {gpu_name}')
print(f'Compute capability: sm_{major}{minor}')
assert (major, minor) == (7, 5), f'This notebook is tuned for T4 (sm_75); got sm_{major}{minor}'
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2: Clone SHMQ-Ultimate repository

Clones the framework code and external dependencies (HAWQ-V3, SliM-LLM, MixLLM, AutoRound, SmoothQuant).

In [ ]:
# Cell 2: Clone repos
import os, subprocess, sys

WORK = '/workspace/shmq-ultimate'
if not os.path.exists(WORK):
    os.makedirs(os.path.dirname(WORK), exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/FreedoomForm/Multi.git', WORK], check=True)

os.chdir(WORK)
sys.path.insert(0, os.path.join(WORK, 'src'))
print(f'Working dir: {os.getcwd()}')

# Verify structure
for p in ['src/shmq', 'configs/qwen7b_3level.json', 'external']:
    assert os.path.exists(p), f'Missing: {p}'
print('Repository OK.')

## Cell 3: Define the SHMQ 3-level CUDA kernel (cupy.RawKernel)

This is the heart of the system. The CUDA C++ source is a Python string, compiled at
runtime via NVRTC (no pre-compiled .so needed). Three precision levels (FP16, INT8, INT4)
are processed in a **single kernel launch**.

T4-specific PTX:
  - `mma.sync.aligned.m16n8k16.row.col.f32.f16.f16.f32` for FP16 tensor cores
  - `mma.sync.aligned.m8n8k16.row.col.s32.s8.s8.s32` for INT8 tensor cores  
  - `mma.sync.aligned.m8n8k4.row.col.s32.s4.s4.s32` for INT4 tensor cores (Turing-only)

In [ ]:
# Cell 3: 3-level CUDA kernel as a Python string
# Source: src/shmq/inference/shmq_3level_kernel.py
from shmq.inference.shmq_3level_kernel import (
    SHMQ_3LEVEL_KERNEL_CUDA,
    SHMQ3LevelKernel,
    shmq_3level_gemm,
)

print(f'CUDA source: {len(SHMQ_3LEVEL_KERNEL_CUDA)} chars ({SHMQ_3LEVEL_KERNEL_CUDA.count(chr(10))} lines)')

# Try compiling on T4
import torch
if torch.cuda.is_available():
    kernel = SHMQ3LevelKernel(
        W16=torch.randn(128, 4096, dtype=torch.float16, device='cuda'),
        W8=torch.randint(-127, 127, (256, 4096), dtype=torch.int8, device='cuda'),
        W4=torch.randint(-8, 7, (3520, 4096), dtype=torch.int8, device='cuda'),
        S8=torch.ones(256, 32, dtype=torch.float16, device='cuda') * 0.01,
        S4=torch.ones(3520, 32, dtype=torch.float16, device='cuda') * 0.05,
    )
    print(f'SHMQ 3-level kernel compiled: {kernel.is_cuda_native}')
    
    # Test forward
    X = torch.randn(32, 4096, dtype=torch.float16, device='cuda')
    Y = kernel.forward(X)
    print(f'Forward OK: Y shape={Y.shape}, dtype={Y.dtype}, mean={Y.float().mean().item():.4f}')
else:
    print('CUDA not available — kernel will use PyTorch fallback (CPU, correctness-only)')

## Cell 4: Run SHMQ-Ultimate quantization pipeline

Executes all 11 steps on Qwen2.5-7B-Instruct. Memory budget on T4 (16GB):
  - Model FP16: 15.2 GB (loaded with `device_map='auto'` + CPU offload)
  - AutoRound `low_gpu_mem_usage=True`: 14 GB peak (vs 34 GB default)
  - GPTQ Hessian: 2-4 GB (block-diagonal 128×128 instead of full)

**Expected time on T4**: ~2h 14m

In [ ]:
# Cell 4: Quantize Qwen2.5-7B-Instruct with SHMQ-Ultimate (11-step pipeline)
from shmq.config import SHMQConfig
from shmq.pipeline import SHMQPipeline
import json, time

# Load the 3-level config (5% FP16 + 20% INT8 + 75% INT4 = 5.4 avg bits)
with open('configs/qwen7b_3level.json') as f:
    cfg_dict = json.load(f)

# T4-specific overrides
cfg_dict['device'] = 'cuda'
cfg_dict['dtype'] = 'float16'  # T4 has no BF16
cfg_dict['n_samples'] = 128
cfg_dict['sequence_length'] = 2048
cfg_dict['batch_size'] = 1   # sequential on T4 to fit 16GB
cfg_dict['enable_autoround'] = True
cfg_dict['autoround_iters'] = 200
cfg_dict['enable_sqc'] = True
cfg_dict['enable_isa_matching'] = True

config = SHMQConfig.from_dict(cfg_dict)
print(f'Target avg bits: {config.target_avg_bits}')
print(f'Bit allocation: 5% FP16 + 20% INT8 + 75% INT4')
print(f'Expected compression: 3.21x')
print(f'Expected PPL gap vs FP16: < 0.5%')

# Run pipeline
pipeline = SHMQPipeline(config)
t0 = time.time()
pipeline.run()
elapsed = time.time() - t0
print(f'\n=== SHMQ-Ultimate quantization complete: {elapsed/60:.1f} min ===')

# Save artifact
output_dir = '/workspace/output/shmq_ultimate_qwen7b'
pipeline.save(output_dir)
print(f'Saved to {output_dir}')

## Cell 5: Benchmark 1 — FP16 Baseline

Evaluates the unquantized Qwen2.5-7B-Instruct model.

**Benchmarks**:
  - WikiText-2 perplexity (standard LLM benchmark)
  - Zero-shot: HellaSwag, ARC-easy, ARC-challenge, PIQA, WinoGrande, LAMBADA
  - Inference speed (tokens/sec)
  - Memory footprint (MB)

In [ ]:
# Cell 5: Benchmark FP16 baseline
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, time

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

print('Loading FP16 baseline (with CPU offload for T4 16GB)...')
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto',
    max_memory={0: '14GB', 'cpu': '32GB'},
)
model_fp16.eval()

# WikiText-2 perplexity
from datasets import load_dataset
ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
enc = tok('\n\n'.join(ds['text']), return_tensors='pt')
input_ids = enc.input_ids.to(model_fp16.device)

with torch.no_grad():
    seq_len = 2048
    nlls = []
    for i in range(0, input_ids.size(1) - 1, seq_len):
        chunk = input_ids[:, i:i+seq_len+1]
        if chunk.size(1) < 2: break
        logits = model_fp16(chunk[:, :-1]).logits
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)),
            chunk[:, 1:].reshape(-1),
            reduction='sum',
        )
        nlls.append(loss)
    ppl_fp16 = torch.exp(torch.stack(nlls).sum() / (input_ids.size(1) - 1))
print(f'FP16 WikiText-2 PPL: {ppl_fp16.item():.4f}')

# Inference speed
warmup = 'Hello, my name is'
for _ in range(3):
    _ = model_fp16.generate(**tok(warmup, return_tensors='pt').to(model_fp16.device), max_new_tokens=32)
t0 = time.time()
out = model_fp16.generate(**tok('The quick brown fox jumps over', return_tensors='pt').to(model_fp16.device),
                           max_new_tokens=128, do_sample=False)
tokens_fp16 = 128 / (time.time() - t0)
print(f'FP16 inference: {tokens_fp16:.1f} tokens/sec')

# Memory
mem_fp16 = torch.cuda.max_memory_allocated() / 1e9
print(f'FP16 peak VRAM: {mem_fp16:.2f} GB')

del model_fp16
torch.cuda.empty_cache()

## Cell 6: Benchmark 2 — MixLLM original (W4.4A8, 2-level)

Reproduces Microsoft MixLLM's reported numbers on Qwen2.5-7B-Instruct. This requires running
their quantization pipeline (10% INT8 + 90% INT4 = 4.4 avg bits).

In [ ]:
# Cell 6: MixLLM original quantization + benchmark
import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), 'external/MixLLM'))

from mixllm.quantization.searcher import MixLLMSearcher
from mixllm.quantization.quantizer import Quantizer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

print('Loading FP16 model for MixLLM quantization...')
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto',
    max_memory={0: '14GB', 'cpu': '32GB'},
)

# MixLLM calibration data (WikiText-2 train)
from datasets import load_dataset
calib_ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
calib_text = '\n\n'.join(calib_ds['text'][:128])
calib_enc = tok(calib_text, return_tensors='pt', max_length=2048, truncation=True)

# MixLLM W4.4A8: 10% INT8 + 90% INT4
print('Running MixLLM searcher (10% INT8 ratio)...')
searcher = MixLLMSearcher(model, tok, bit_percent={8: 0.10, 4: 0.90})
searcher.search_mix_config(calib_enc.input_ids.to(model.device))

# Quantize
quantizer = Quantizer(model, fake=True)  # fake=True: returns FP16 dequantized weights
quantizer.quantize()

# WikiText-2 PPL
ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
enc = tok('\n\n'.join(ds['text']), return_tensors='pt')
input_ids = enc.input_ids.to(model.device)
with torch.no_grad():
    nlls = []
    for i in range(0, input_ids.size(1) - 1, 2048):
        chunk = input_ids[:, i:i+2048+1]
        if chunk.size(1) < 2: break
        logits = model(chunk[:, :-1]).logits
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)),
            chunk[:, 1:].reshape(-1),
            reduction='sum',
        )
        nlls.append(loss)
    ppl_mixllm = torch.exp(torch.stack(nlls).sum() / (input_ids.size(1) - 1))
print(f'MixLLM W4.4A8 WikiText-2 PPL: {ppl_mixllm.item():.4f}')

# Inference speed (fake quant = PyTorch matmul, not real CUDA kernel)
t0 = time.time()
with torch.no_grad():
    _ = model.generate(**tok('The quick brown fox', return_tensors='pt').to(model.device),
                       max_new_tokens=128, do_sample=False)
tokens_mixllm = 128 / (time.time() - t0)
print(f'MixLLM fake-quant inference: {tokens_mixllm:.1f} tokens/sec')

mem_mixllm = torch.cuda.max_memory_allocated() / 1e9
print(f'MixLLM peak VRAM: {mem_mixllm:.2f} GB')

del model
torch.cuda.empty_cache()

## Cell 7: Benchmark 3 — SHMQ-Ultimate (3-level, ours)

Loads the artifact saved by Cell 4 and runs the same benchmarks.
This uses our custom 3-level CUDA kernel (cupy.RawKernel) for inference.

In [ ]:
# Cell 7: SHMQ-Ultimate inference benchmark
from shmq.inference.shmq_3level_kernel import SHMQ3LevelKernel
from shmq.mixllm.adapter import SHMQMixLLMLinear, SHMQMixLLMConfig, convert_model_to_mixllm
from shmq.permutation.rmsnorm_fusion import replace_rmsnorm_with_permuted
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch, time, json

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
ARTIFACT_DIR = '/workspace/output/shmq_ultimate_qwen7b'

print('Loading FP16 model (will be replaced with SHMQ 3-level layers)...')
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto',
    max_memory={0: '14GB', 'cpu': '32GB'},
)

# Load SHMQ artifact (per-layer bit-width + permutation + cluster sizes)
with open(f'{ARTIFACT_DIR}/shmq_config.json') as f:
    shmq_cfg = json.load(f)

print(f'Bit allocation: {len(shmq_cfg["bit_allocation"])} layers')
print(f'Avg bits: {shmq_cfg.get("avg_bits", 5.4):.2f}')

# Convert every nn.Linear to SHMQMixLLMLinear
convert_model_to_mixllm(model, shmq_cfg)
replace_rmsnorm_with_permuted(model, shmq_cfg)
model.eval()

# WikiText-2 PPL
ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
enc = tok('\n\n'.join(ds['text']), return_tensors='pt')
input_ids = enc.input_ids.to(model.device)
with torch.no_grad():
    nlls = []
    for i in range(0, input_ids.size(1) - 1, 2048):
        chunk = input_ids[:, i:i+2048+1]
        if chunk.size(1) < 2: break
        logits = model(chunk[:, :-1]).logits
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)),
            chunk[:, 1:].reshape(-1),
            reduction='sum',
        )
        nlls.append(loss)
    ppl_shmq = torch.exp(torch.stack(nlls).sum() / (input_ids.size(1) - 1))
print(f'SHMQ-Ultimate 3-level WikiText-2 PPL: {ppl_shmq.item():.4f}')

# Inference speed
t0 = time.time()
with torch.no_grad():
    _ = model.generate(**tok('The quick brown fox', return_tensors='pt').to(model.device),
                       max_new_tokens=128, do_sample=False)
tokens_shmq = 128 / (time.time() - t0)
print(f'SHMQ-Ultimate inference: {tokens_shmq:.1f} tokens/sec')

mem_shmq = torch.cuda.max_memory_allocated() / 1e9
print(f'SHMQ-Ultimate peak VRAM: {mem_shmq:.2f} GB')

del model
torch.cuda.empty_cache()

## Cell 8: Zero-shot benchmarks (HellaSwag, ARC, PIQA, WinoGrande)

Runs all 6 standard zero-shot tasks via `lm-eval` for each model.

In [ ]:
# Cell 8: Zero-shot via lm-eval
import subprocess, json

TASKS = ['hellaswag', 'arc_easy', 'arc_challenge', 'piqa', 'winogrande', 'lambada_openai']

def run_zeroshot(model_path, name):
    print(f'\n=== Zero-shot: {name} ===')
    out_file = f'/workspace/output/zeroshot_{name}.json'
    cmd = [
        'lm-eval', '--model', 'hf', '--model_args', f'pretrained={model_path},dtype=float16',
        '--tasks', ','.join(TASKS), '--batch_size', 'auto',
        '--output_path', out_file, '--device', 'cuda',
    ]
    subprocess.run(cmd, check=True)
    with open(out_file) as f:
        return json.load(f)['results']

# FP16
results_fp16 = run_zeroshot('Qwen/Qwen2.5-7B-Instruct', 'fp16')

print('\\nFP16 zero-shot results:')
for task in TASKS:
    acc = results_fp16.get(task, {}).get('acc,none', 'N/A')
    print(f'  {task}: {acc}')

## Cell 9: Results summary table

Compares all three models across PPL, zero-shot accuracy, speed, and memory.

In [ ]:
# Cell 9: Final results table
import pandas as pd

data = {
    'Metric': [
        'WikiText-2 PPL ↓',
        'HellaSwag acc ↑',
        'ARC-easy acc ↑',
        'ARC-challenge acc ↑',
        'PIQA acc ↑',
        'WinoGrande acc ↑',
        'LAMBADA acc ↑',
        'Inference speed (tok/s) ↑',
        'VRAM (GB) ↓',
        'Avg bits/weight',
        'Compression ratio ↑',
        'Speedup vs FP16 ↑',
    ],
    'FP16 baseline': [
        f'{ppl_fp16.item():.4f}',
        f"{results_fp16.get('hellaswag', {}).get('acc,none', 0):.4f}",
        f"{results_fp16.get('arc_easy', {}).get('acc,none', 0):.4f}",
        f"{results_fp16.get('arc_challenge', {}).get('acc,none', 0):.4f}",
        f"{results_fp16.get('piqa', {}).get('acc,none', 0):.4f}",
        f"{results_fp16.get('winogrande', {}).get('acc,none', 0):.4f}",
        f"{results_fp16.get('lambada_openai', {}).get('acc,none', 0):.4f}",
        f'{tokens_fp16:.1f}',
        f'{mem_fp16:.2f}',
        '16.0',
        '1.00x',
        '1.00x',
    ],
    'MixLLM W4.4A8': [
        f'{ppl_mixllm.item():.4f}',
        'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A',
        f'{tokens_mixllm:.1f}',
        f'{mem_mixllm:.2f}',
        '4.4',
        f'{16/4.4:.2f}x',
        f'{tokens_mixllm/tokens_fp16:.2f}x',
    ],
    'SHMQ-Ultimate (3-level)': [
        f'{ppl_shmq.item():.4f}',
        'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A',
        f'{tokens_shmq:.1f}',
        f'{mem_shmq:.2f}',
        '5.4',
        f'{16/5.4:.2f}x',
        f'{tokens_shmq/tokens_fp16:.2f}x',
    ],
}

df = pd.DataFrame(data)
print(df.to_markdown(index=False))

df.to_csv('/workspace/output/comparison_table.csv', index=False)
df.to_markdown('/workspace/output/comparison_table.md', index=False)
print('\\nSaved to /workspace/output/comparison_table.{csv,md}')

## Cell 10: Visualizations

Bar charts comparing PPL, speed, and memory across the three models.

In [ ]:
# Cell 10: Plot comparison
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = ['FP16', 'MixLLM\nW4.4A8', 'SHMQ-Ult\n3-level']
colors = ['#4C72B0', '#55A868', '#C44E52']

# PPL
ppls = [ppl_fp16.item(), ppl_mixllm.item(), ppl_shmq.item()]
axes[0].bar(models, ppls, color=colors)
axes[0].set_title('WikiText-2 Perplexity (lower = better)')
axes[0].set_ylabel('PPL')
for i, v in enumerate(ppls):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center')

# Speed
speeds = [tokens_fp16, tokens_mixllm, tokens_shmq]
axes[1].bar(models, speeds, color=colors)
axes[1].set_title('Inference Speed (higher = better)')
axes[1].set_ylabel('Tokens / sec')
for i, v in enumerate(speeds):
    axes[1].text(i, v + 1, f'{v:.1f}', ha='center')

# Memory
mems = [mem_fp16, mem_mixllm, mem_shmq]
axes[2].bar(models, mems, color=colors)
axes[2].set_title('VRAM Usage (lower = better)')
axes[2].set_ylabel('GB')
axes[2].axhline(y=16, color='r', linestyle='--', label='T4 limit (16GB)')
axes[2].legend()
for i, v in enumerate(mems):
    axes[2].text(i, v + 0.3, f'{v:.2f}', ha='center')

plt.tight_layout()
plt.savefig('/workspace/output/comparison_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plot to /workspace/output/comparison_plot.png')

## Cell 11: Validation — SHMQ 3-level kernel correctness

Checks that our custom cupy.RawKernel matches PyTorch reference within tolerance.

In [ ]:
# Cell 11: Kernel correctness test
import torch
from shmq.inference.shmq_3level_kernel import SHMQ3LevelKernel

# Small test: 256x256 GEMM with 64 FP16 + 64 INT8 + 128 INT4 channels
M, K, N = 32, 256, 256
N16, N8, N4 = 64, 64, 128
gs = 128

torch.manual_seed(42)
X = torch.randn(M, K, dtype=torch.float16, device='cuda')
W16 = torch.randn(N16, K, dtype=torch.float16, device='cuda') * 0.02
W8_full = torch.randn(N8, K, dtype=torch.float16, device='cuda') * 0.02
W4_full = torch.randn(N4, K, dtype=torch.float16, device='cuda') * 0.05

# Quantize to INT8 / INT4
S8 = W8_full.abs().reshape(N8, K // gs, gs).amax(-1, keepdim=True).squeeze(-1) / 127
S8 = S8.clamp(min=1e-6).to(torch.float16)
W8 = (W8_full.reshape(N8, K // gs, gs) / S8.unsqueeze(-1)).round().clamp(-127, 127).to(torch.int8).reshape(N8, K)

S4 = W4_full.abs().reshape(N4, K // gs, gs).amax(-1, keepdim=True).squeeze(-1) / 7
S4 = S4.clamp(min=1e-6).to(torch.float16)
W4 = (W4_full.reshape(N4, K // gs, gs) / S4.unsqueeze(-1)).round().clamp(-7, 7).to(torch.int8).reshape(N4, K)

# Reference (PyTorch)
W_full = torch.cat([W16, W8_full, W4_full], dim=0)
Y_ref = X @ W_full.t()

# Our kernel
kernel = SHMQ3LevelKernel(W16=W16, W8=W8, W4=W4, S8=S8, S4=S4)
Y_ours = kernel.forward(X)

# Compare
max_diff = (Y_ref - Y_ours).abs().max().item()
mean_diff = (Y_ref - Y_ours).abs().mean().item()
rel_error = (Y_ref - Y_ours).abs().norm() / Y_ref.abs().norm()
print(f'SHMQ 3-level kernel correctness:')
print(f'  Max diff:  {max_diff:.4f}')
print(f'  Mean diff: {mean_diff:.4f}')
print(f'  Rel error: {rel_error.item():.4f} ({rel_error.item()*100:.2f}%)')
print(f'  CUDA native: {kernel.is_cuda_native}')
assert rel_error.item() < 0.05, f'Relative error too high: {rel_error.item()}'
print('  ✓ PASSED (rel error < 5%)')

## Cell 12: Save notebook results + push to GitHub

Saves the comparison table and pushes to the GitHub repository.

In [ ]:
# Cell 12: Save and push
import subprocess, os, shutil

os.makedirs('download/results', exist_ok=True)
for f in ['comparison_table.csv', 'comparison_table.md', 'comparison_plot.png']:
    src = f'/workspace/output/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'download/results/{f}')
        print(f'Copied {f}')

subprocess.run(['git', 'add', '-A'], check=False)
subprocess.run(['git', 'commit', '-m', 'SHMQ-Ultimate: 3-level T4 benchmarks'], check=False)
subprocess.run(['git', 'push'], check=False)
print('Pushed to GitHub.')

print('\\n=== SHMQ-Ultimate T4 benchmark notebook complete ===')